In [0]:
%pip install duckduckgo-search -q

In [0]:
%pip install ddgs -q

In [0]:
import json
import requests
import re
import time
import warnings
from datetime import datetime
from pyspark.sql import SparkSession
from concurrent.futures import ThreadPoolExecutor, as_completed
from ddgs import DDGS

# 💡 FIX 1: Suppress annoying SSL Socket Warnings from DDGS
warnings.filterwarnings("ignore", category=ResourceWarning)

NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# ==========================================
# 1. Safe Human-Like Web Search (Anti-Block)
# ==========================================
def human_like_search(company, title):
    print(f"   🕵️‍♂️ Gently searching web for: {company}...")
    queries = [f'"{company}" IT staffing OR careers']
    combined_snippets = []
    
    try:
        # 💡 FIX 2: Added a slight delay so DuckDuckGo doesn't block our IPs
        time.sleep(1.5) 
        with DDGS() as ddgs:
            # Just fetching 1 result to keep it lightweight and avoid bans
            results = list(ddgs.text(queries[0], max_results=1))
            for res in results:
                combined_snippets.append(f"Source URL: {res.get('href')}\nInfo: {res.get('body')}")
    except Exception as e:
        # If DDGS blocks us, don't crash. Just pass empty context to AI.
        pass 
        
    if combined_snippets:
        return "\n\n".join(combined_snippets)
    else:
        return "No web footprint found. It might be a small staffing agency or masked vendor."

# ==========================================
# 2. Bench Sales Focused AI Validation
# ==========================================
def validate_job_with_ai(company_name, job_title, job_description, web_context):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    # 💡 FIX 3: Taught AI the exact US IT Bench Sales ecosystem!
    prompt = f"""
    You are an expert US IT Bench Sales Recruiter evaluating a job requirement.
    
    Agency/Company Posting the Job: {company_name}
    Role: {job_title}
    Job Description: 
    {job_description}
    
    Web Search Context: {web_context}
    
    CRITICAL BENCH SALES RULES:
    1. In the US IT market, staffing agencies (like Vacancy Global Pro, Remote Zest Jobs, etc.) frequently post jobs hiding the end-client name as 'reputed company' or 'our client'. THIS IS 100% NORMAL and VALID for C2C (Corp-to-Corp) or W2 contracts.
    2. DO NOT reject a job just because the company lacks a Wikipedia page or uses a generic staffing template.
    3. REJECT ONLY IF: It asks the candidate for money, contains completely irrelevant non-IT spam, or is clearly a phishing scam.
    4. Categorize job type strictly as one of: [C2C, W2, Full-Time, Contract, Unknown]. If it's a staffing agency, it is likely C2C or Contract.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "is_valid": true,
        "job_type": "Contract/C2C",
        "hr_email": "hr@company.com or null",
        "reasoning": "Standard US IT Staffing requirement, client name is masked but skills match perfectly."
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            start_idx = content.find('{')
            end_idx = content.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_json = content[start_idx:end_idx+1]
                return json.loads(clean_json)
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "AI Parsing Failed"}
    except Exception as e:
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "Exception occurred"}

# ==========================================
# 3. Safe Multi-Threaded Execution
# ==========================================
def process_single_job(row_dict):
    job_id = row_dict['raw_job_id']
    company = row_dict['company_name']
    title = row_dict['job_title']
    desc = row_dict['job_description']
    apply_link = row_dict['apply_link']
    
    web_context = human_like_search(company, title)
    ai_result = validate_job_with_ai(company, title, desc, web_context)
    
    return {
        "job_id": job_id,
        "company": company,
        "title": title,
        "apply_link": apply_link,
        "ai_result": ai_result
    }

def run_job_validation_pipeline():
    print("🚀 Starting Bench-Sales Optimized Validation Pipeline...")
    
    # Let's pull 20 jobs for a strong test
    pending_jobs_df = spark.sql("""
        SELECT * FROM jobs_automation_db.default.raw_jobs_staging 
        WHERE validation_status = 'Pending' LIMIT 20
    """)
    
    pending_count = pending_jobs_df.count()
    if pending_count == 0:
        print("✅ No pending jobs to validate today. Pipeline up to date!")
        return
        
    print(f"🔎 Found {pending_count} pending jobs. Processing safely to avoid API bans...\n")
    
    pending_jobs = [row.asDict() for row in pending_jobs_df.collect()]
    validated_records = []
    processed_job_ids = []
    
    # 💡 FIX 4: Reduced max_workers to 2. It will be slightly slower, but 100% stable without DDGS SSL crashes!
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_to_job = {executor.submit(process_single_job, job): job for job in pending_jobs}
        
        for future in as_completed(future_to_job):
            result = future.result()
            job_id = result['job_id']
            ai_result = result['ai_result']
            company = result['company']
            title = result['title']
            
            if ai_result and ai_result.get("is_valid"):
                print(f"   ✔️ ACCEPTED! {title} @ {company} | Type: {ai_result.get('job_type')}")
                validated_records.append({
                    "job_id": job_id,
                    "company_name": company,
                    "job_title": title,
                    "job_type": ai_result.get("job_type", "Unknown"),
                    "is_valid": True,
                    "hr_email": ai_result.get("hr_email"),
                    "apply_url": result['apply_link'],
                    "validated_date": datetime.now()
                })
            else:
                reason = ai_result.get("reasoning", "No reason provided") if ai_result else "AI Failed"
                print(f"   ❌ REJECTED! {title} @ {company} | Reason: {reason}")
                
            processed_job_ids.append(f"'{job_id}'")
            
    if len(validated_records) > 0:
        valid_df = spark.createDataFrame(validated_records)
        ordered_columns = [
            "job_id", "company_name", "job_title", "job_type", 
            "is_valid", "hr_email", "apply_url", "validated_date"
        ]
        valid_df = valid_df.select(*ordered_columns)
        valid_df.write.mode("append").insertInto("jobs_automation_db.default.validated_jobs_master")
        print(f"\n💾 Saved {len(validated_records)} BENCH SALES VALID jobs to master table.")
    
    if processed_job_ids:
        ids_string = ",".join(processed_job_ids)
        spark.sql(f"""
            UPDATE jobs_automation_db.default.raw_jobs_staging 
            SET validation_status = 'Processed' 
            WHERE raw_job_id IN ({ids_string})
        """)
        print("🔄 Updated raw_jobs_staging status to 'Processed'.")

run_job_validation_pipeline()

In [0]:
%sql
select * from jobs_automation_db.default.validated_jobs_master

In [0]:
import json
import requests
import re
from datetime import datetime
from pyspark.sql import SparkSession
from concurrent.futures import ThreadPoolExecutor, as_completed
from ddgs import DDGS # Updated package to avoid warnings

# API Key
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# ==========================================
# 1. Human-Like Live Web Search
# ==========================================
def human_like_search(company, title):
    print(f"   🕵️‍♂️ Human-like search for: {company}...")
    
    # ఒక మనిషి వెతికే తరహాలో 2 డిఫరెంట్ సెర్చ్ క్వెరీస్
    queries = [
        f'"{company}" "{title}" careers OR jobs',
        f'"{company}" company Wikipedia OR official website'
    ]
    
    combined_snippets = []
    try:
        with DDGS() as ddgs:
            for q in queries:
                # ప్రతి క్వెరీకి టాప్ 2 రిజల్ట్స్ తీసుకుంటున్నాం
                results = list(ddgs.text(q, max_results=2))
                for res in results:
                    combined_snippets.append(f"Source URL: {res.get('href')}\nInformation: {res.get('body')}")
    except Exception as e:
        print(f"   ⚠️ Web search obstacle for {company}: {str(e)}")
        
    if combined_snippets:
        return "\n\n".join(combined_snippets)
    else:
        return "No valid footprint found on Wikipedia or official career pages."

# ==========================================
# 2. AI Validation with Live Context
# ==========================================
def validate_job_with_ai(company_name, job_title, job_description, web_context):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert IT Technical Recruiter and Job Validator.
    Analyze the following Job Description AND the Live Web Search Results to extract key information.
    
    Company: {company_name}
    Role: {job_title}
    
    Job Description (May contain masked words like 'reputed company'): 
    {job_description}
    
    LIVE WEB SEARCH RESULTS (Contains Wikipedia/Official Career page footprints):
    {web_context}
    
    Instructions:
    1. Validate the job. If the JD looks like spam BUT the Web Search Results show a real Wikipedia page or official careers page for this company, mark it as VALID (true). If both look fake/spammy, mark it false.
    2. Determine the Job Type. Strictly choose ONE from: [W2, C2C, Contract, Full-Time, Unknown]. Look at the web context for clues.
    3. Look carefully for any HR or Recruiter email address.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "is_valid": true,
        "job_type": "Full-Time",
        "hr_email": "hr@company.com",
        "reasoning": "Brief 1-sentence reason referencing the web search (e.g., 'Found official Wikipedia page and careers portal')."
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            start_idx = content.find('{')
            end_idx = content.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_json = content[start_idx:end_idx+1]
                return json.loads(clean_json)
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "AI Parsing Failed"}
    except Exception as e:
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "Exception occurred"}

# ==========================================
# 3. Threaded Single Job Processor
# ==========================================
def process_single_job(row_dict):
    job_id = row_dict['raw_job_id']
    company = row_dict['company_name']
    title = row_dict['job_title']
    desc = row_dict['job_description']
    apply_link = row_dict['apply_link']
    
    # 1. Human-Like Search
    web_context = human_like_search(company, title)
    
    # 2. AI Validation
    ai_result = validate_job_with_ai(company, title, desc, web_context)
    
    return {
        "job_id": job_id,
        "company": company,
        "title": title,
        "apply_link": apply_link,
        "ai_result": ai_result
    }

# ==========================================
# 4. Multi-Threaded Execution Pipeline
# ==========================================
def run_job_validation_pipeline():
    print("🚀 Starting MULTI-THREADED Live Web Job Validation Pipeline...")
    
    pending_jobs_df = spark.sql("""
        SELECT * FROM jobs_automation_db.default.raw_jobs_staging 
        WHERE validation_status = 'Pending' LIMIT 15
    """)
    
    pending_count = pending_jobs_df.count()
    if pending_count == 0:
        print("✅ No pending jobs to validate today. Pipeline up to date!")
        return
        
    print(f"🔎 Found {pending_count} pending jobs. Firing up Threads for parallel human-like analysis...\n")
    
    # DataFrame rows ని Dictionary లాగా మార్చుకోవడం (Threading కి ఈజీగా ఉంటుంది)
    pending_jobs = [row.asDict() for row in pending_jobs_df.collect()]
    
    validated_records = []
    processed_job_ids = []
    
    # 💡 MAGIC: ThreadPoolExecutor తో ఒకేసారి 5 జాబ్స్ ని పారలల్ గా రన్ చేస్తున్నాం
    with ThreadPoolExecutor(max_workers=5) as executor:
        # Submit all jobs to the executor
        future_to_job = {executor.submit(process_single_job, job): job for job in pending_jobs}
        
        for future in as_completed(future_to_job):
            result = future.result()
            job_id = result['job_id']
            ai_result = result['ai_result']
            company = result['company']
            title = result['title']
            
            if ai_result and ai_result.get("is_valid"):
                print(f"   ✔️ VALID! {title} @ {company} | Type: {ai_result.get('job_type')}")
                validated_records.append({
                    "job_id": job_id,
                    "company_name": company,
                    "job_title": title,
                    "job_type": ai_result.get("job_type", "Unknown"),
                    "is_valid": True,
                    "hr_email": ai_result.get("hr_email"),
                    "apply_url": result['apply_link'],
                    "validated_date": datetime.now()
                })
            else:
                reason = ai_result.get("reasoning", "No reason provided") if ai_result else "AI Failed"
                print(f"   ❌ INVALID! {title} @ {company} | Reason: {reason}")
                
            processed_job_ids.append(f"'{job_id}'")
            
    # 5. Database Updates
    if len(validated_records) > 0:
        valid_df = spark.createDataFrame(validated_records)
        ordered_columns = [
            "job_id", "company_name", "job_title", "job_type", 
            "is_valid", "hr_email", "apply_url", "validated_date"
        ]
        valid_df = valid_df.select(*ordered_columns)
        valid_df.write.mode("append").insertInto("jobs_automation_db.default.validated_jobs_master")
        print(f"\n💾 Saved {len(validated_records)} GOLDEN jobs to validated_jobs_master table.")
    
    if processed_job_ids:
        ids_string = ",".join(processed_job_ids)
        spark.sql(f"""
            UPDATE jobs_automation_db.default.raw_jobs_staging 
            SET validation_status = 'Processed' 
            WHERE raw_job_id IN ({ids_string})
        """)
        print("🔄 Updated raw_jobs_staging status to 'Processed'.")

run_job_validation_pipeline()

In [0]:
%sql
select * from jobs_automation_db.default.validated_jobs_master

In [0]:
import json
import requests
import re
from datetime import datetime
from pyspark.sql import SparkSession
from duckduckgo_search import DDGS

# API Key
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# ==========================================
# 1. Live Web Search Function
# ==========================================
def live_web_search(company, title):
    print(f"   🌐 Surfing the web/LinkedIn for: {company} {title}...")
    query = f'"{company}" "{title}" job linkedin'
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if results:
                snippets = []
                for res in results:
                    snippets.append(f"Title: {res.get('title')}\nSnippet: {res.get('body')}\nLink: {res.get('href')}")
                return "\n\n".join(snippets)
    except Exception as e:
        print(f"   ⚠️ Web search failed: {str(e)}")
    return "No live web results found."

# ==========================================
# 2. AI Validation with Live Context
# ==========================================
def validate_job_with_ai(company_name, job_title, job_description, web_context):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert IT Technical Recruiter and Job Validator.
    Analyze the following Job Description AND the Live Web Search Results to extract key information.
    
    Company: {company_name}
    Role: {job_title}
    
    Job Description (May contain masked words like 'reputed company'): 
    {job_description}
    
    LIVE WEB SEARCH RESULTS (Use this to verify if the job/company actually exists):
    {web_context}
    
    Instructions:
    1. Determine if this job is valid. Even if the JD is masked with 'reputed company', if the Web Search Results show a real LinkedIn or company career page for this role, mark it as VALID (true).
    2. Determine the Job Type. Strictly choose ONE from: [W2, C2C, Contract, Full-Time, Unknown]. Look at the web context for clues.
    3. Look carefully for any HR or Recruiter email address.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "is_valid": true,
        "job_type": "Full-Time",
        "hr_email": "hr@company.com",
        "reasoning": "Brief 1-sentence reason referencing the web search or JD."
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            start_idx = content.find('{')
            end_idx = content.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_json = content[start_idx:end_idx+1]
                return json.loads(clean_json)
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "AI Parsing Failed"}
    except Exception as e:
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "Exception occurred"}

# ==========================================
# 3. Execution Pipeline
# ==========================================
def run_job_validation_pipeline():
    print("🚀 Starting LIVE Web Job Validation Pipeline...")
    
    pending_jobs_df = spark.sql("""
        SELECT * FROM jobs_automation_db.default.raw_jobs_staging 
        WHERE validation_status = 'Pending' LIMIT 5
    """)
    
    pending_count = pending_jobs_df.count()
    if pending_count == 0:
        print("✅ No pending jobs to validate today. Pipeline up to date!")
        return
        
    print(f"🔎 Found {pending_count} pending jobs. AI is analyzing them using live web search...\n")
    
    pending_jobs = pending_jobs_df.collect()
    
    validated_records = []
    processed_job_ids = []
    
    for row in pending_jobs:
        job_id = row['raw_job_id']
        company = row['company_name']
        title = row['job_title']
        desc = row['job_description']
        apply_link = row['apply_link']
        
        print(f"⚙️ Validating: {title} @ {company}...")
        
        # 1. Do a live search first!
        web_context = live_web_search(company, title)
        
        # 2. Call AI with live data
        ai_result = validate_job_with_ai(company, title, desc, web_context)
        
        if ai_result and ai_result.get("is_valid"):
            print(f"   ✔️ VALID! Type: {ai_result.get('job_type')} | HR Email: {ai_result.get('hr_email')}")
            
            validated_records.append({
                "job_id": job_id,
                "company_name": company,
                "job_title": title,
                "job_type": ai_result.get("job_type", "Unknown"),
                "is_valid": True,
                "hr_email": ai_result.get("hr_email"),
                "apply_url": apply_link,
                "validated_date": datetime.now()
            })
        else:
            reason = ai_result.get("reasoning", "No reason provided") if ai_result else "AI Failed"
            print(f"   ❌ INVALID/SKIPPED! Reason: {reason}")
            
        processed_job_ids.append(f"'{job_id}'")
        
    if len(validated_records) > 0:
        valid_df = spark.createDataFrame(validated_records)
        ordered_columns = [
            "job_id", "company_name", "job_title", "job_type", 
            "is_valid", "hr_email", "apply_url", "validated_date"
        ]
        valid_df = valid_df.select(*ordered_columns)
        valid_df.write.mode("append").insertInto("jobs_automation_db.default.validated_jobs_master")
        print(f"\n💾 Saved {len(validated_records)} GOLDEN jobs to validated_jobs_master table.")
    
    if processed_job_ids:
        ids_string = ",".join(processed_job_ids)
        spark.sql(f"""
            UPDATE jobs_automation_db.default.raw_jobs_staging 
            SET validation_status = 'Processed' 
            WHERE raw_job_id IN ({ids_string})
        """)
        print("🔄 Updated raw_jobs_staging status to 'Processed'.")

run_job_validation_pipeline()

In [0]:
import json
import requests
import re
from datetime import datetime
from pyspark.sql import SparkSession

# API Key
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

def validate_job_with_ai(company_name, job_title, job_description):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert IT Technical Recruiter and Job Validator.
    Analyze the following Job Description and extract key information.
    
    Company: {company_name}
    Role: {job_title}
    Job Description: 
    {job_description}
    
    Instructions:
    1. Determine if this job is valid. (Mark false if it looks like spam, asks for money, or is extremely poorly written).
    2. Determine the Job Type. Strictly choose ONE from: [W2, C2C, Contract, Full-Time, Unknown].
    3. Look carefully for any HR or Recruiter email address mentioned in the text. If none found, return null.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "is_valid": true,
        "job_type": "Full-Time",
        "hr_email": "hr@company.com",
        "reasoning": "Brief 1-sentence reason on why it is valid/invalid."
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            # Bulletproof JSON extraction
            start_idx = content.find('{')
            end_idx = content.rfind('}')
            if start_idx != -1 and end_idx != -1:
                clean_json = content[start_idx:end_idx+1]
                return json.loads(clean_json)
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "AI Parsing Failed"}
    except Exception as e:
        print(f"❌ AI Error: {str(e)}")
        return {"is_valid": False, "job_type": "Unknown", "hr_email": None, "reasoning": "Exception occurred"}

def run_job_validation_pipeline():
    print("🚀 Starting Job Validation Pipeline...")
    
    # 1. Fetch only 'Pending' jobs from staging
    pending_jobs_df = spark.sql("""
        SELECT * FROM jobs_automation_db.default.raw_jobs_staging 
        WHERE validation_status = 'Pending' LIMIT 5
    """)
    
    pending_count = pending_jobs_df.count()
    if pending_count == 0:
        print("✅ No pending jobs to validate today. Pipeline up to date!")
        return
        
    print(f"🔎 Found {pending_count} pending jobs. AI is analyzing them now...\n")
    
    pending_jobs = pending_jobs_df.collect()
    
    validated_records = []
    processed_job_ids = []
    
    # 2. Process each job through NVIDIA AI
    for row in pending_jobs:
        job_id = row['raw_job_id']
        company = row['company_name']
        title = row['job_title']
        desc = row['job_description']
        apply_link = row['apply_link']
        
        print(f"⚙️ Validating: {title} @ {company}...")
        
        # Call AI for validation
        ai_result = validate_job_with_ai(company, title, desc)
        
        if ai_result and ai_result.get("is_valid"):
            print(f"   ✔️ VALID! Type: {ai_result.get('job_type')} | HR Email: {ai_result.get('hr_email')}")
            
            validated_records.append({
                "job_id": job_id,
                "company_name": company,
                "job_title": title,
                "job_type": ai_result.get("job_type", "Unknown"),
                "is_valid": True,
                "hr_email": ai_result.get("hr_email"),
                "apply_url": apply_link,
                "validated_date": datetime.now()
            })
        else:
            reason = ai_result.get("reasoning", "No reason provided") if ai_result else "AI Failed"
            print(f"   ❌ INVALID/SKIPPED! Reason: {reason}")
            
        # Keep track of processed IDs to update raw table
        processed_job_ids.append(f"'{job_id}'")
        
    # 3. Store valid jobs into the Golden Table
    if len(validated_records) > 0:
        valid_df = spark.createDataFrame(validated_records)
        
        ordered_columns = [
            "job_id", "company_name", "job_title", "job_type", 
            "is_valid", "hr_email", "apply_url", "validated_date"
        ]
        valid_df = valid_df.select(*ordered_columns)
        
        # Insert into Master Table
        valid_df.write.mode("append").insertInto("jobs_automation_db.default.validated_jobs_master")
        print(f"\n💾 Saved {len(validated_records)} GOLDEN jobs to validated_jobs_master table.")
    
    # 4. Update the raw table status to 'Processed'
    if processed_job_ids:
        ids_string = ",".join(processed_job_ids)
        update_query = f"""
            UPDATE jobs_automation_db.default.raw_jobs_staging 
            SET validation_status = 'Processed' 
            WHERE raw_job_id IN ({ids_string})
        """
        spark.sql(update_query)
        print("🔄 Updated raw_jobs_staging status to 'Processed'.")

# Execute the Validation Pipeline
run_job_validation_pipeline()